In [10]:
from libraries import *
from functions import *
from bayesian import *
import sankey
from importlib import reload
reload(sankey)
from sankey import *

In [11]:
adult_dataset= pd.read_csv("adult.txt")
adult_dataset.head()
adult_dataset.shape


(32561, 15)

In [12]:
df_cs = adult_dataset.copy()


df_cs["X"] = (df_cs["race"] == "White").astype(int)

x_0 = 0
x_1 = 1

df_cs["Y"] = (df_cs["income"] == ">50K").astype(int)

#Grouping age 
df_cs["age_group"] = pd.cut(
    df_cs["age"],
    bins=[0, 25, 45, 65, 100],
    labels=["young", "mid", "senior", "elder"],
    include_lowest=True,
)
z_cols = ["age_group", "sex", "native.country"]      # confounders
w_cols = [ "relationship","occupation",]          # mediators




Data preprocessing for readability

In [13]:
cols = ["X", "Y"] + z_cols + w_cols
df_cs = df_cs[
    ~(
        df_cs[cols].isna()        #NaN
        | (df_cs[cols] == "?")    #string "?"
    ).any(axis=1)
]

df_cs.shape

(30162, 18)

In [14]:
df_cs.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income,X,Y,age_group
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K,1,0,elder
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K,1,0,senior
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K,1,0,mid
5,34,Private,216864,HS-grad,9,Divorced,Other-service,Unmarried,White,Female,0,3770,45,United-States,<=50K,1,0,mid
6,38,Private,150601,10th,6,Separated,Adm-clerical,Unmarried,White,Male,0,3770,40,United-States,<=50K,1,0,mid


In [15]:

W_values = list(map(tuple, df_cs[w_cols].drop_duplicates().values))

print(df_cs.shape)

effects = compute_effects_multi(
    df=df_cs,
    x0=x_0,
    x1=x_1,
    y=1,
    W_values=W_values,
    x_col="X",
    y_col="Y",
    w_cols=w_cols,
    z_cols=z_cols,
    w_order=None,
    z_order=None,
)

(30162, 18)


/Users/alessiaberarducci/Desktop/causal-ai-fairness/causalfairness/bayesian.py:214: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/Users/alessiaberarducci/Desktop/causal-ai-fairness/causalfairness/bayesian.py:214: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [16]:
effects

{'tv': 0.08264842400686287,
 'te': 0.08775590706746396,
 'te_linear': 0.03351030941954666,
 'se': -0.005107483060601095,
 'se_x1': -0.0005335889216601248,
 'se_x0': 0.00457389413894097,
 'de': -0.018702653527167965,
 'ie': -0.051991653017115655,
 'ie_f': 0.05221296294671463,
 'ie_decomp': {},
 'se_decomp': {},
 'ie_decomp_interval': {'relationship': (-0.03532648514298681,
   -0.01883797612076407),
  'occupation': (-0.03800064884036358, -0.022074507307465102)},
 'se_decomp_interval': {'age_group': (-0.0038116707726570235,
   0.003968831214232783),
  'sex': (0.01955353244177682, 0.02462658635605419),
  'native.country': (-0.031442815265079346, -0.022145031700961278)}}

In [21]:
plot_effect_sankey_percent(
    te=effects["te"],          
    se=effects["se"],
    ie=effects["ie"],
    de=effects["de"],
    title="Adult: sex → income – percent of |TV|"
)
